# TripAdvisor Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/tripadvisor_reviews.py`.

Covers:
1. **API call** – fetching reviews via TripAdvisor Content API (full review text)
2. Ollama classification – topic & sentiment detection
3. Manual review input – adding reviews that the API can't fetch

TripAdvisor blocks direct web scraping (DataDome CAPTCHA) and Google Maps only
shows truncated snippets of TripAdvisor reviews. The Content API is the only
reliable source for full review text.

**Requirements:** Ollama running locally with `mistral:7b`, `TRIPADVISOR_API_KEY` set.

In [1]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

Project root: /home/laurabquintas/projects/reputation-analyzer


In [ ]:
from src.sites import tripadvisor_reviews as tr
import requests

api_key = os.getenv('TRIPADVISOR_API_KEY')

location_id = tr.LOCATION_IDS.get(tr.ANANEA_HOTEL)
print(f'Hotel: {tr.ANANEA_HOTEL}')
print(f'Location ID: {location_id}')
print(f'API key: {"set" if api_key else "NOT SET – set TRIPADVISOR_API_KEY to fetch reviews"}')

Hotel: Ananea Castelo Suites Hotel
Location ID: 33299137
API key: set


## 1. API Call – Single Page, English

Requires `TRIPADVISOR_API_KEY`. Returns max 5 reviews per page.

In [4]:
# Fetch a single page of English reviews (requires API key)
if api_key:
    reviews_en, total_en = tr.ta_get_reviews_page(location_id, api_key, language='en', offset=0)
    print(f'English reviews returned: {len(reviews_en)}')
    print(f'Total English reviews available: {total_en}')
    print()
    for r in reviews_en:
        print(f"  [{r.get('id')}] {r.get('rating')}★ – {r.get('title')} ({r.get('published_date', '')[:10]})")
else:
    print('TRIPADVISOR_API_KEY not set – skip API call')

English reviews returned: 3
Total English reviews available: 0

  [1037605287] 5★ – Lovely quiet hotel (2025-11-03)
  [1033353426] 5★ – Fabulous Modern Hotel (2025-10-04)
  [1033043012] 5★ – Wonderful week with wonderful staff (2025-10-02)


## 2. API Call – All Languages with Pagination

In [ ]:
# Fetch across all default languages with pagination (requires API key)
if api_key:
    all_reviews = tr.ta_get_reviews(location_id, api_key)
    print(f'Total unique reviews fetched: {len(all_reviews)}')
    print()
    for r in sorted(all_reviews, key=lambda x: x.get('published_date', ''), reverse=True):
        lang = r.get('_language', '?')
        country = tr._extract_country(r)
        print(f"  [{r.get('id')}] {r.get('rating')}★ lang={lang} country={country or '?'} – {r.get('title')} ({r.get('published_date', '')[:10]})")
else:
    all_reviews = []
    print('TRIPADVISOR_API_KEY not set – skip API call')

In [6]:
# Inspect raw API response for first review
if all_reviews:
    print(json.dumps(all_reviews[0], indent=2, ensure_ascii=False))
else:
    print('No API reviews to inspect')

{
  "id": 1037605287,
  "lang": "en",
  "location_id": 33299137,
  "published_date": "2025-11-03T11:15:52Z",
  "rating": 5,
  "helpful_votes": 0,
  "rating_image_url": "https://www.tripadvisor.com/img/cdsi/img2/ratings/traveler/s5.0-66827-5.svg",
  "url": "https://www.tripadvisor.com/ShowUserReviews-g189112-d33299137-r1037605287-Reviews-Castelo_Suites_Hotel_ananea-Albufeira_Faro_District_Algarve.html?m=66827#review1037605287",
  "text": "We travelled as a family of four with 2 children 9 and 11 and really enjoyed our time at this hotel.\n\nThe hotel is in a quiet area with a few restaurants and bars within walking distance. There are a couple of beaches nearby (Praia do Evaristo and Praia do Castelo) both with restaurants and well worth a visit. I would recommend hiring a car, there is ample free parking available.\n\nWe stayed in a family suite with twin beds in one room and a king in the other. Each had its own door but were accessed by a single door at the front which could be locke

## 3. Ollama Classification – Test

Uses API reviews normalised to a common format:
`{id, rating, text, title, author_name, published_date}`.

In [7]:
# Check Ollama availability
from src.classification import classify_review, is_ollama_available

ollama_ok = is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')

Ollama available: True
Models: ['mistral:7b']


In [ ]:
# Build review list from API results (common format for classification)
reviews_for_classification = []

if 'all_reviews' in dir() and all_reviews:
    for r in all_reviews:
        reviews_for_classification.append({
            'id': str(r.get('id', '')),
            'rating': r.get('rating'),
            'title': r.get('title', ''),
            'text': r.get('text', ''),
            'published_date': r.get('published_date', ''),
            'author_name': (r.get('user') or {}).get('username', ''),
            'country': tr._extract_country(r) or 'Unknown',
            'travel_date': r.get('travel_date', ''),
            'trip_type': r.get('trip_type', '') or 'Unknown',
            'subratings': r.get('subratings', {}),
            'helpful_votes': r.get('helpful_votes', 0),
        })
    print(f'Using {len(reviews_for_classification)} reviews from TripAdvisor API')
else:
    print('No reviews available – set TRIPADVISOR_API_KEY and run Section 2 first')

if reviews_for_classification:
    r0 = reviews_for_classification[0]
    print(f'Sample: {r0.get("author_name", "?")} from {r0.get("country", "?")} – {r0.get("text", "")[:80]}...')

In [9]:
# Classify a single review (pick the first with text)
test_review = next((r for r in reviews_for_classification if r.get('text')), None)
if test_review and ollama_ok:
    print(f"Review by: {test_review.get('author_name', 'Unknown')}")
    title = test_review.get('title', '')
    if title:
        print(f"Title: {title}")
    print(f"Text: {test_review['text'][:300]}...")
    print()
    topics = classify_review(test_review['text'])
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('No review with text found or Ollama not available')

Review by: PippaNel
Title: Lovely quiet hotel
Text: We travelled as a family of four with 2 children 9 and 11 and really enjoyed our time at this hotel.

The hotel is in a quiet area with a few restaurants and bars within walking distance. There are a couple of beaches nearby (Praia do Evaristo and Praia do Castelo) both with restaurants and well wor...

Classification result (6 topics):
  🟢 employees = positive
  🔴 commodities = negative
  🟢 commodities = positive
  🟢 return = positive
  🟢 meals = positive
  🔴 employees = negative


In [ ]:
# Classify ALL fetched reviews, save to JSON (overwrites existing reviews by ID)
from datetime import datetime

if ollama_ok and reviews_for_classification:
    json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
    existing = tr.load_reviews(json_path)
    existing_by_id = {r['id']: r for r in existing}

    for r in reviews_for_classification:
        text = r.get('text', '')
        if not text:
            continue
        topics = classify_review(text)

        review_id = str(r.get('id', ''))
        pub_date = r.get('published_date', '')
        if len(pub_date) == 10:
            pub_date = f"{pub_date}T00:00:00Z"

        record = {
            'id': review_id,
            'hotel': tr.ANANEA_HOTEL,
            'location_id': location_id or '',
            'source': 'tripadvisor',
            'rating': r.get('rating'),
            'title': r.get('title', ''),
            'text': text,
            'published_date': pub_date,
            'author_name': r.get('author_name', ''),
            'country': r.get('country', '') or 'Unknown',
            'travel_date': r.get('travel_date', ''),
            'trip_type': r.get('trip_type', '') or 'Unknown',
            'subratings': r.get('subratings', {}),
            'helpful_votes': r.get('helpful_votes', 0),
            'scraped_date': datetime.now().strftime('%Y-%m-%d'),
            'topics': topics,
            'classified': True,
        }

        action = 'updated' if review_id in existing_by_id else 'added'
        existing_by_id[review_id] = record

        stars = '\u2605' * int(r.get('rating') or 0)
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{stars} [{action}] {r.get('title') or r.get('author_name', 'N/A')[:45]:45s} \u2192 {topic_str or '(none detected)'}")

    all_records = list(existing_by_id.values())
    tr.save_reviews(all_records, json_path)
    print(f'\nSaved {len(all_records)} reviews to {json_path}')
else:
    print('Ollama not available or no reviews \u2013 skip classification')

In [ ]:
# Debug: see the raw Ollama response for a specific review
# Change the index to test different reviews
DEBUG_INDEX = 0

if ollama_ok and reviews_for_classification:
    review = reviews_for_classification[DEBUG_INDEX]
    text = review.get('text', '')
    print(f"Review by: {review.get('author_name', 'Unknown')}")
    title = review.get('title', '')
    if title:
        print(f"Title: {title}")
    print(f"Full text:\n{text}\n")
    print('--- Sending to Ollama ---')

    from src.classification import _parse_classification

    prompt = f"""You are a hotel review analyst. Read the review below carefully and identify ALL topics mentioned, even briefly. Pay special attention to complaints, cons, criticisms, and suggestions for improvement – these are NEGATIVE.

TOPICS (use these exact keys):
- employees: any mention of staff, service, friendliness, helpfulness, reception, concierge, team, waiters, management
- commodities: amenities, facilities, pool, gym, spa, room features, wifi, parking, fridge, toiletries, TV, air conditioning, balcony, shuttle, iron, entertainment, music
- comfort: room comfort, bed quality, noise, quiet, space, temperature, room size, mattress, pillow, decor, ambiance, construction noise, view
- cleaning: cleanliness, hygiene, tidiness, housekeeping, spotless, dirty, stains, towels changed, room serviced
- quality_price: value for money, pricing, worth, cost, overpriced, good deal, expensive, cheap, affordable, half board value
- meals: food, breakfast, restaurant, dining, bar, drinks, buffet, dinner, lunch, cuisine, menu, chef, kitchen, snacks, repetitive food, variety
- return: whether the guest would return, come back, visit again, recommend, revisit, not return, wouldn't go back

RULES:
1. You MUST check each topic one by one. Go through the review sentence by sentence.
2. A single review CAN and OFTEN DOES have both positive AND negative for the SAME topic. For example breakfast can be praised (positive) but also called repetitive (negative).
3. Even brief or indirect mentions count (e.g. "rooms were cleaned daily" = cleaning positive).
4. If a topic is described positively, mark it positive. If negatively, mark it negative.
5. Complaints, cons, "could be better", "didn't work well", "wish they had", suggestions for improvement = NEGATIVE. Do NOT skip these.
6. Output ONLY a JSON array. No explanation, no markdown.

Now analyze this review:
\"\"\"{text}\"\"\"

JSON array:"""

    payload = {'model': 'mistral:7b', 'prompt': prompt, 'stream': False,
               'options': {'temperature': 0.1, 'num_predict': 512}}
    resp = requests.post('http://localhost:11434/api/generate', json=payload, timeout=120)
    raw = resp.json().get('response', '')
    print(f'Raw Ollama response:\n{raw}')
    print()
    parsed = _parse_classification(raw)
    print(f'Parsed: {json.dumps(parsed, indent=2)}')
else:
    print('Ollama not available or no reviews')

## 4. Manual Review Input

Since the API only returns 5 reviews per page, you can manually add reviews below.
Fill in the fields and run the cell to add them to the JSON file.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'reviewer_name': 'John D.',                    # reviewer name (used for ID)
    'rating': 5,                                    # 1-5
    'title': 'Amazing stay',                        # review title
    'text': 'Paste the full review text here...',   # full review text
    'published_date': '2026-03-01',                 # YYYY-MM-DD
    'trip_type': 'Couples',                         # Family, Couples, Solo, Business, Friends
    'country': 'Unknown',                           # reviewer country (or Unknown)
}
# ========================================

# Generate unique ID from name + date + title
id_seed = f"{manual_review['reviewer_name']}_{manual_review['published_date']}_{manual_review['title']}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

# Format published_date to match API format
pub_date = manual_review['published_date']
if len(pub_date) == 10:  # YYYY-MM-DD
    pub_date = f"{pub_date}T00:00:00Z"

record = {
    'id': review_id,
    'hotel': tr.ANANEA_HOTEL,
    'location_id': location_id,
    'rating': manual_review['rating'],
    'title': manual_review['title'],
    'text': manual_review['text'],
    'published_date': pub_date,
    'author_name': manual_review['reviewer_name'],
    'country': manual_review.get('country', '') or 'Unknown',
    'travel_date': '',
    'trip_type': manual_review.get('trip_type', '') or 'Unknown',
    'subratings': {},
    'helpful_votes': 0,
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
    'source': 'manual',
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
if ollama_ok and record['text'] and record['text'] != 'Paste the full review text here...':
    topics = classify_review(record['text'])
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '🟢' if t['sentiment'] == 'positive' else '🔴'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('Ollama not available or no text – skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
existing = tr.load_reviews(json_path)

# Check for duplicates
existing_ids = {r['id'] for r in existing}
if record['id'] in existing_ids:
    print(f'⚠️  Review {record["id"]} already exists – skipping')
else:
    existing.append(record)
    tr.save_reviews(existing, json_path)
    print(f'✅ Saved! Total reviews: {len(existing)}')

## 5. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once.

In [ ]:
import hashlib
# Add multiple reviews at once
batch_reviews = [
    {
        'reviewer_name': 'Maria S.',
        'rating': 4,
        'title': 'Good hotel, cold pool',
        'text': 'Replace with actual review text...',
        'published_date': '2026-02-15',
        'trip_type': 'Family',
        'country': 'Unknown',
    },
    # Add more reviews here...
]


json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
existing = tr.load_reviews(json_path)
existing_ids = {r['id'] for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['reviewer_name']}_{mr['published_date']}_{mr['title']}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]
    
    if rid in existing_ids:
        print(f'⚠️  Skip duplicate: {mr["title"]}')
        continue
    
    pub_date = mr['published_date']
    if len(pub_date) == 10:
        pub_date = f"{pub_date}T00:00:00Z"
    
    rec = {
        'id': rid,
        'hotel': tr.ANANEA_HOTEL,
        'location_id': location_id,
        'rating': mr['rating'],
        'title': mr['title'],
        'text': mr['text'],
        'published_date': pub_date,
        'author_name': mr['reviewer_name'],
        'country': mr.get('country', '') or 'Unknown',
        'travel_date': '',
        'trip_type': mr.get('trip_type', '') or 'Unknown',
        'subratings': {},
        'helpful_votes': 0,
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
        'source': 'manual',
    }
    
    # Classify if Ollama available
    if ollama_ok and rec['text'] and 'Replace with' not in rec['text']:
        try:
            topics = classify_review(rec['text'])
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')
    
    existing.append(rec)
    existing_ids.add(rid)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"✅ {mr['title']} → {topic_str or '(unclassified)'}")

if added:
    tr.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

## 6. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [ ]:
json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
reviews = tr.load_reviews(json_path)

# Find reviews that are "classified" but have 0 topics – likely LLM failures
needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics') and r.get('text')]
# Also find unclassified reviews
unclassified = [r for r in reviews if not r.get('classified') and r.get('text')]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = classify_review(r['text'])
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            author = r.get('title') or r.get('author_name', 'N/A')
            print(f"  ✅ {author[:40]} → {topic_str or '(still none)'}")
        except Exception as e:
            author = r.get('title') or r.get('author_name', 'N/A')
            print(f"  ❌ {author[:40]} → Error: {e}")
    
    tr.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

## 7. Reclassify ALL Reviews

Re-runs classification on **every** review with text. Use this after changing
the classification prompt in `src/classification.py`.

In [ ]:
json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
reviews = tr.load_reviews(json_path)

with_text = [r for r in reviews if r.get('text')]
print(f'Total reviews: {len(reviews)}')
print(f'Reviews with text (will reclassify): {len(with_text)}')

if ollama_ok and with_text:
    for i, r in enumerate(with_text, 1):
        try:
            old_topics = r.get('topics', [])
            new_topics = classify_review(r['text'])
            r['topics'] = new_topics
            r['classified'] = True

            old_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in old_topics)
            new_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in new_topics)
            changed = '🔄' if old_str != new_str else '✅'
            author = r.get('title') or r.get('author_name', 'N/A')
            print(f"  {changed} [{i}/{len(with_text)}] {author[:40]}")
            if old_str != new_str:
                print(f"       old: {old_str or '(none)'}")
                print(f"       new: {new_str or '(none)'}")
        except Exception as e:
            author = r.get('title') or r.get('author_name', 'N/A')
            print(f"  ❌ [{i}/{len(with_text)}] {author[:40]} → Error: {e}")

    tr.save_reviews(reviews, json_path)
    print(f'\nDone. Reclassified {len(with_text)} reviews. Saved {len(reviews)} total.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('No reviews with text found.')

## 8. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
reviews = tr.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if r.get("source") == "manual")}')
print()

# Country breakdown
country_counts = {}
for r in reviews:
    c = r.get('country', 'Unknown') or 'Unknown'
    country_counts[c] = country_counts.get(c, 0) + 1
if country_counts:
    print('Country breakdown:')
    for c, n in sorted(country_counts.items(), key=lambda x: -x[1]):
        print(f'  {c}: {n}')
    print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')